# KRAS docking screen &mdash; DiffDock + GNINA

GPU tier of the pipeline. Takes an **already-gated** ligand list, docks it against one
protein target, and writes results in the exact shape `dock_tox.run_docking()` consumes,
so the output drops straight back into `connector.py`.

**Before you run:**

1. Notebook settings &rarr; Accelerator = **GPU T4 x2**. NOT P100 &mdash; Kaggle's torch 2.10+cu128 dropped Pascal (sm_60) support, so a P100 reports itself available but cannot launch a kernel.
2. Notebook settings &rarr; Internet = **On** (needs phone verification on your Kaggle account;
   without it `pip install` and the ESM weight download both fail)
3. Upload your gated ligand list as a Kaggle Dataset, or edit `LIGANDS` below to point at it

**Runtime budget.** Kaggle gives ~30 GPU-hours/week and a 12-hour session cap. At roughly
30-60s per ligand this notebook checkpoints after every batch and resumes from disk, so a
session dying at 12h costs you one batch, not the run.

**Scope.** This is the expensive tier. Do not feed it a raw library &mdash; run `prepare_ligands.py`
on your laptop first to gate for chemical reality, then a cheap rigid-docking pass, and send
only the survivors here.

## 1 &nbsp; Environment check

Confirm the GPU, driver, and disk before installing anything.

In [4]:

import shutil, subprocess, os, sys

ok = True

# --- GPU -------------------------------------------------------------------
if shutil.which("nvidia-smi") is None:
    ok = False
    print("NO GPU ATTACHED")
    print("  nvidia-smi is not present, so this session has no accelerator.")
    print("  Fix: right-hand panel -> Session options -> Accelerator -> GPU P100")
    print("       (Kaggle restarts the session; nothing is lost at this point.)")
else:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

# --- torch -----------------------------------------------------------------
try:
    import torch
    print(f"torch          {torch.__version__}")
    print(f"cuda (torch)   {torch.version.cuda}")
    print(f"cuda available {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        cap  = torch.cuda.get_device_capability(0)
        sm   = f"sm_{cap[0]}{cap[1]}"
        arches = torch.cuda.get_arch_list()
        print(f"device         {name}")
        print(f"compute cap    {sm}")
        print(f"vram           {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

        # cuda.is_available() is NOT enough. Kaggle's torch 2.10+cu128 dropped Pascal
        # (sm_60), so a P100 reports available=True but cannot launch a single kernel -
        # DiffDock silently falls back to CPU at 21 min/ligand, or crashes.
        if sm not in arches:
            ok = False
            print(f"\n  GPU NOT SUPPORTED BY THIS TORCH BUILD")
            print(f"  {name} is {sm}; this torch supports: {' '.join(arches)}")
            print(f"  Fix: Session options -> Accelerator -> GPU T4 x2  (T4 is sm_75)")
            print(f"       Do NOT use P100 on this image.")
    else:
        ok = False
except ImportError:
    ok = False
    print("torch is not installed in this image")

# --- internet --------------------------------------------------------------
try:
    import urllib.request
    urllib.request.urlopen("https://pypi.org", timeout=8)
    print("internet       reachable")
except Exception:
    ok = False
    print("NO INTERNET")
    print("  Fix: right-hand panel -> Session options -> Internet -> On")
    print("       Needs phone verification on your Kaggle account.")
    print("       Without it the DiffDock clone, ESM weights and GNINA download all fail.")

# --- resources -------------------------------------------------------------
free = shutil.disk_usage("/kaggle/working").free
ram  = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
print(f"disk free      {free/1e9:.1f} GB")
print(f"ram            {ram/1e9:.1f} GB")
print(f"python         {sys.version.split()[0]}")

print()
print("READY" if ok else "NOT READY - fix the items above before running the next cell.")


Mon Aug 31 23:06:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2 &nbsp; Install

PyG's compiled extensions must match the exact torch + CUDA build Kaggle ships, so the wheel
index URL is derived from the running torch rather than hardcoded.

In [5]:
import torch, subprocess, sys

TORCH = torch.__version__.split("+")[0]
CUDA  = "cpu" if torch.version.cuda is None else "cu" + torch.version.cuda.replace(".", "")
PYG_URL = f"https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html"

if CUDA == "cpu":
    raise RuntimeError(
        "CPU-only torch build (torch.version.cuda is None) - the accelerator is not attached.\n\n"
        "STOP HERE. Do not install CPU wheels: DiffDock takes ~21 MINUTES PER LIGAND on CPU\n"
        "versus ~30-60 seconds on a P100, so a real screen is impossible.\n\n"
        "Fix: Session options -> Accelerator -> GPU P100. The session restarts and clears\n"
        "/kaggle/working, so re-run this notebook from cell 1."
    )

print("torch          ", torch.__version__)
print("PyG wheel index", PYG_URL, "\n")

def pip(*args, optional=False):
    """Install one spec. Report and continue rather than killing the cell."""
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                       capture_output=True, text=True)
    name = args[0]
    if r.returncode == 0:
        print(f"  ok    {name}")
    else:
        tail = (r.stderr or r.stdout).strip().splitlines()[-3:]
        print(f"  FAIL  {name}")
        for l in tail:
            print(f"        {l[:110]}")
        if not optional:
            raise RuntimeError(f"required package failed: {name}")

# compiled PyG extensions - DiffDock needs scatter and cluster
print("PyG extensions:")
for pkg in ("torch_scatter", "torch_sparse", "torch_cluster"):
    pip(pkg, "-f", PYG_URL)
pip("torch_geometric")

# DiffDock's stack. Unpinned where the old docking_requirements.txt pins predate Python 3.12.
#   biopython : 1.79 has no cp312 wheel and fails to build from source. Modern versions
#               keep the PDB.PDBParser / PDBIO / Select API this pipeline uses.
#   fair-esm  : plain, NOT [esmfold]. DiffDock needs ESM2 embeddings only; the esmfold
#               extra drags in openfold + deepspeed, which will not build here.
print("\nDiffDock stack:")
pip("e3nn==0.5.0")
pip("fair-esm")
pip("biopython")
pip("networkx<3")
pip("rdkit")
pip("pyyaml")
pip("pandas")
pip("scipy")
pip("wget")
pip("spyrmsd", optional=True)   # RMSD scoring only, not on the inference path
pip("prody", optional=True)     # not on the inference path

print("\nversions:")
import importlib
for m in ("torch_geometric", "torch_scatter", "torch_cluster", "e3nn", "esm", "Bio", "networkx", "rdkit"):
    try:
        mod = importlib.import_module(m)
        print(f"  {m:<18} {getattr(mod, '__version__', 'installed')}")
    except ImportError:
        print(f"  {m:<18} MISSING")

torch           2.10.0+cu128
PyG wheel index https://data.pyg.org/whl/torch-2.10.0+cu128.html 

PyG extensions:
  ok    torch_scatter
  ok    torch_sparse
  ok    torch_cluster
  ok    torch_geometric

DiffDock stack:
  ok    e3nn==0.5.0
  ok    fair-esm
  ok    biopython
  ok    networkx<3
  ok    rdkit
  ok    pyyaml
  ok    pandas
  ok    scipy
  ok    wget
  ok    spyrmsd
  ok    prody

versions:
  torch_geometric    2.8.0.post1
  torch_scatter      2.1.2+pt210cu128
  torch_cluster      1.6.3+pt210cu128
  e3nn               0.5.0
  esm                2.0.0
  Bio                1.88
  networkx           2.8.8
  rdkit              2026.03.5


## 3 &nbsp; DiffDock

Pinned to **v1.1**, which *is* DiffDock-L &mdash; the release is titled
"Version 1.1: DiffDock-L". You get the current model and your existing
`default_inference_args.yaml` resolves unmodified.

Weights are not in the repo; they ship as a 130 MB release asset that unzips
to exactly the `workdir/v1.1/...` layout the config expects.

The last block fixes a name collision that only bites on fat images like
Kaggle's: DiffDock's local `datasets/` directory is shadowed by the
preinstalled HuggingFace `datasets` package.

In [6]:
import os, subprocess, pathlib, sys

DIFFDOCK_DIR = "/kaggle/working/DiffDock"
WEIGHTS_URL  = "https://github.com/gcorso/DiffDock/releases/download/v1.1/diffdock_models.zip"

# v1.1 IS DiffDock-L - the release is titled "Version 1.1: DiffDock-L". Pinning here gets
# the current model AND keeps default_inference_args.yaml working unmodified.
if not os.path.exists(DIFFDOCK_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "v1.1",
                    "https://github.com/gcorso/DiffDock.git", DIFFDOCK_DIR], check=True)

os.chdir(DIFFDOCK_DIR)

# Weights are NOT in the repo. They ship as a release asset and unzip to exactly the
# layout default_inference_args.yaml expects:
#     workdir/v1.1/score_model/best_ema_inference_epoch_model.pt
#     workdir/v1.1/confidence_model/best_model_epoch75.pt
wd = pathlib.Path(DIFFDOCK_DIR) / "workdir" / "v1.1"
score_ckpt = wd / "score_model" / "best_ema_inference_epoch_model.pt"

if not score_ckpt.exists():
    wd.mkdir(parents=True, exist_ok=True)
    print("downloading weights (130 MB)...")
    subprocess.run(["wget", "-q", "-O", "/kaggle/working/diffdock_models.zip", WEIGHTS_URL], check=True)
    subprocess.run(["unzip", "-q", "-o", "/kaggle/working/diffdock_models.zip", "-d", str(wd)], check=True)
    os.remove("/kaggle/working/diffdock_models.zip")

ok = True
for rel, min_mb in [("score_model/best_ema_inference_epoch_model.pt", 100),
                    ("confidence_model/best_model_epoch75.pt", 15),
                    ("score_model/model_parameters.yml", 0),
                    ("confidence_model/model_parameters.yml", 0)]:
    p = wd / rel
    if not p.exists():
        print(f"  MISSING  {rel}"); ok = False
    else:
        mb = p.stat().st_size / 1e6
        bad = mb < min_mb
        print(f"  {mb:8.1f} MB  {rel}{'   <-- TOO SMALL' if bad else ''}")
        ok &= not bad

print("\ncwd:", os.getcwd())
print("weights ready" if ok else "WEIGHTS BAD - inference will fail")

# --- namespace collision fix ------------------------------------------------
# DiffDock's local datasets/ has NO __init__.py, so it is only a namespace package.
# Python resolves regular packages before namespace ones regardless of sys.path order,
# and Kaggle preinstalls HuggingFace `datasets` as a regular package - so
# `from datasets.process_mols import ...` inside inference.py hits the wrong package
# and dies with ModuleNotFoundError. Never happens on a lean box; always happens here.
import os, subprocess, sys, importlib

for pkg in ("datasets", "confidence", "models", "utils"):
    d = os.path.join(DIFFDOCK_DIR, pkg)
    init = os.path.join(d, "__init__.py")
    if os.path.isdir(d) and not os.path.exists(init):
        open(init, "w").close()
        print(f"  created {pkg}/__init__.py")

# Belt and braces: drop the shadowing package. Nothing in this pipeline uses it.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "datasets"],
               capture_output=True, text=True)

for m in [m for m in list(sys.modules) if m == "datasets" or m.startswith("datasets.")]:
    del sys.modules[m]
importlib.invalidate_caches()

# Verify by resolving exactly the way inference.py does, from DiffDock's own directory.
r = subprocess.run([sys.executable, "-c",
                    "from datasets.process_mols import write_mol_with_coords; print('import ok')"],
                   cwd=DIFFDOCK_DIR, capture_output=True, text=True)
out = (r.stdout or r.stderr).strip()
print(out)
if "import ok" not in out:
    raise RuntimeError("DiffDock imports still broken - do not continue")

     121.1 MB  score_model/best_ema_inference_epoch_model.pt
      19.3 MB  confidence_model/best_model_epoch75.pt
       0.0 MB  score_model/model_parameters.yml
       0.0 MB  confidence_model/model_parameters.yml

cwd: /kaggle/working/DiffDock
weights ready
import ok


## 4 &nbsp; GNINA

The only published binary is `gnina.cuda12.8.static` &mdash; Linux x86_64 + CUDA. This is exactly
why this stage cannot run on an Apple Silicon laptop, and why it belongs in a notebook.

CUDA minor-version compatibility means the 12.8 build runs on any driver 525+, which covers
Kaggle. If it fails, the fallback cell drops to CPU scoring (much slower but correct).

In [7]:
import os, subprocess, stat

GNINA = "/kaggle/working/gnina"
URL = "https://github.com/gnina/gnina/releases/latest/download/gnina.cuda12.8.static"

if not os.path.exists(GNINA):
    subprocess.run(["wget", "-q", "-O", GNINA, URL], check=True)
    os.chmod(GNINA, os.stat(GNINA).st_mode | stat.S_IEXEC)

r = subprocess.run([GNINA, "--version"], capture_output=True, text=True)
print(r.stdout or r.stderr)

GNINA_CPU_ONLY = r.returncode != 0
if GNINA_CPU_ONLY:
    print("\n!! GNINA failed to start on GPU. Will pass --cpu to every call.")
    print("   Check the driver version in nvidia-smi above against the CUDA 12.8 build.")
else:
    print("GNINA ready")

gnina v1.3.3 master:6fe1ce2   Built Jun 30 2026.

GNINA ready


## 5 &nbsp; Target and inputs

One target per run. The four KRAS variants and PI3K are pre-filled from `cross_docking.py`.

In [8]:
TARGETS = {
    "KRAS_G12C": dict(
        pdb_url="https://files.rcsb.org/download/8AFB.pdb",
        sequence="GMTEYKLVVVGACGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETCLLDILDTAGQEEYSAMRDQYMRTGEGFLCVFAINNTKSFEDIHHYREQIKRVKDSEDVPMVLVGNKSDLPSRTVDTKQAQDLARSYGIPFIETSAKTRQGVDDAFYTLVREIRKHKEK"),
    "KRAS_G12D": dict(
        pdb_url="https://files.rcsb.org/download/7RT5.pdb",
        sequence="GMTEYKLVVVGADGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETSLLDILDTAGQEEYSAMRDQYMRTGEGFLLVFAINNTKSFEDIHHYREQIKRVKDSEDVPMVLVGNKSDLPSRTVDTKQAQDLARSYGIPFIETSAKTRQGVDDAFYTLVREIRKHKEK"),
    "KRAS_G12V": dict(
        pdb_url="https://files.rcsb.org/download/7C40.pdb",
        sequence="MTEYKLVVVGAVGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETCLLDILDTAGQEEYSAMRDQYMRTGEGFLCVFAINNTKSFEDIHHYREQIKRVKDSEDVPMVLVGNKCDLPSRTVDTKQAQDLARSYGIPFIETSAKTRQGVDDAFYTLVREIRKHKEHHHHHH"),
    "KRAS_G13D": dict(
        pdb_url="https://files.rcsb.org/download/6E6F.pdb",
        sequence="MTEYKLVVVGAGDVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETCLLDILDTAGQEEYSAMRDQYMRTGEGFLCVFAINNTKSFEDIHHYREQIKRVKDSEDVPMVLVGNKCDLPSRTVDTKQAQDLARSYGIPFIETSAKTRQGVDDAFYTLVREIRKH"),
}

TARGET = "KRAS_G12C"

# Ligand input: a JSON list of SMILES, or a .smi / .csv with one SMILES per line.
# Produce this with prepare_ligands.py on your laptop - do NOT paste a raw library here.
#
# LEAVE AS None FOR YOUR FIRST RUN. With no ligand file the notebook docks only the
# controls below - two molecules, a known answer, ~5 min of GPU. That proves DiffDock,
# ESM, and GNINA all work before you spend quota on a real screen.
LIGANDS = None
# LIGANDS = "/kaggle/input/gated-ligands/ligands.json"   # <- after you upload a Dataset

# Positive controls. A control is only meaningful if it is COGNATE to the receptor -
# co-crystallised in the very structure you are docking into. The G12C switch-II pocket
# is induced-fit: it does not exist in apo KRAS, it is carved open by whichever inhibitor
# bound it. So each PDB's pocket is shaped by ITS ligand, and docking a different drug in
# is a cross-docking problem where scores degrade for reasons that are not about binding.
#
# 8AFB = BI-0474 (ligand LXD).  Sotorasib's cognate is 6OIM.  Adagrasib's is 6UT0.
#
# Measured on 8AFB: Adagrasib -9.85 (tolerates the pocket), Sotorasib -3.80 (does not).
# That reproduces the 2025 campaign (-9.12 / -5.00) and is a property of the method on
# this receptor, not a pipeline fault.
COGNATE = {
    "8AFB": ("BI-0474", "C[C@H]1CN(CCN1c2cc(cc(n2)c3nc(on3)[C@]4(CCCc5c4c(c(s5)N)C#N)C)N6CCN(CC6)C(=O)C=C)C"),
}

CONTROLS = {
    # cognate to 8AFB - this one MUST score well or the method is broken here
    "BI-0474":   "C[C@H]1CN(CCN1c2cc(cc(n2)c3nc(on3)[C@]4(CCCc5c4c(c(s5)N)C#N)C)N6CCN(CC6)C(=O)C=C)C",
    # non-cognate references, kept for continuity with the 2025 numbers
    "Sotorasib": "CC(C)C1=NC=CC(C)=C1N1C(=O)N=C(N2CCN(C[C@@H]2C)C(=O)C=C)C2=CC(F)=C(N=C12)C1=C(O)C=CC=C1F",
    "Adagrasib": "[H][C@@]1(COC2=NC3=C(CCN(C3)C3=CC=CC4=C3C(Cl)=CC=C4)C(=N2)N2CCN(C(=O)C(F)=C)[C@@]([H])(CC#N)C2)CCCN1C",
}

BATCH_SIZE          = 25     # ligands per checkpoint. Smaller = less lost when a session dies.
SAMPLES_PER_COMPLEX = 10     # DiffDock poses per ligand
INFERENCE_STEPS     = 20
OUT = "/kaggle/working/results"

import os, json
os.makedirs(OUT, exist_ok=True)
print(f"target  {TARGET}")
print(f"output  {OUT}")

target  KRAS_G12C
output  /kaggle/working/results


## 6 &nbsp; The gate

Runs again here, on purpose. GPU time is the expensive resource and this is the cheap check
that stops you spending it on molecules that cannot be made.

**Covalent exemption.** BRENK flags acrylamide as `Michael_acceptor_1`, which rejects both
Sotorasib and Adagrasib &mdash; the warhead is the mechanism, not a liability. For covalent
programmes the exemption is on by default. Turn it off for non-covalent targets.

In [9]:
from rdkit import Chem, RDLogger
from rdkit.Chem import rdMolDescriptors, FilterCatalog
RDLogger.DisableLog("rdApp.*")

COVALENT_PROGRAMME = True     # True for KRAS G12C. False elsewhere.

UNSTABLE = {
    "hydrazine N-N":     "[NX3;!$(N=*)]-[NX3;!$(N=*)]",
    "triazane N-N-N":    "[NX3;!$(N=*)]-[NX3;!$(N=*)]-[NX3;!$(N=*)]",
    "peroxide O-O":      "[OX2]-[OX2]",
    "epoxide":           "C1OC1",
    "aziridine":         "C1NC1",
    "azo N=N":           "[NX2]=[NX2]",
    "N-oxide / nitroso": "[NX3]-[OX2H0,OX1]",
    "boron":             "[B]",
    "strained alkene":   "[CX3]1=[CX3][CX4]1",
    "1,2-diketone":      "[CX3](=O)[CX3](=O)",
}
INCOMPATIBLE = [
    ("[NX3;!$(N=*)]-[NX3;!$(N=*)]", "[CX3H1](=O)[#6]", "hydrazine + aldehyde condense"),
    ("[NX3;!$(N=*)]-[NX3;!$(N=*)]", "C1OC1",           "hydrazine opens the epoxide"),
    ("[NX3;H2][CX4]",               "[CX3H1](=O)[#6]", "amine + aldehyde condense"),
]
_U  = {k: Chem.MolFromSmarts(v) for k, v in UNSTABLE.items()}
_IC = [(Chem.MolFromSmarts(a), Chem.MolFromSmarts(b), w) for a, b, w in INCOMPATIBLE]

_p = FilterCatalog.FilterCatalogParams()
_p.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
_p.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
_CAT = FilterCatalog.FilterCatalog(_p)

COVALENT_OK = {"Michael_acceptor_1"}

def gate(smiles):
    """Return None if the molecule passes, else the reason it failed."""
    m = Chem.MolFromSmiles(smiles)
    if m is None:
        return "unparseable"
    for name, patt in _U.items():
        if m.HasSubstructMatch(patt):
            return f"unstable group: {name}"
    for a, b, why in _IC:
        if m.HasSubstructMatch(a) and m.HasSubstructMatch(b):
            return f"self-reactive: {why}"
    hit = _CAT.GetFirstMatch(m)
    if hit:
        desc = hit.GetDescription()
        if not (COVALENT_PROGRAMME and desc in COVALENT_OK):
            return f"catalog: {desc}"
    if rdMolDescriptors.CalcNumAromaticRings(m) == 0:
        return "no aromatic ring"
    return None

# self-test: the two approved drugs must pass a covalent programme
for n, s in CONTROLS.items():
    print(f"  {n:<11} {gate(s) or 'PASS'}")

  BI-0474     PASS
  Sotorasib   PASS
  Adagrasib   PASS


## 7 &nbsp; Docking core

Same logic as your `docking.py`, unchanged where it worked.

In [10]:
import os, re, csv, glob, yaml, shutil, subprocess, uuid
from copy import deepcopy
import pandas as pd
from Bio import PDB
import wget


def read_sdf_as_string(path):
    with open(path) as f:
        return f.read()


def remove_water_and_ligands(input_pdb, output_pdb, retain_hets=("ZN", "MG")):
    structure = PDB.PDBParser(QUIET=True).get_structure("protein", input_pdb)
    io = PDB.PDBIO(); io.set_structure(structure)

    class Sel(PDB.Select):
        def accept_residue(self, residue):
            if residue.id[0] == " ":
                return True
            return residue.id[0] == "H" and residue.resname.strip() in retain_hets

    io.save(output_pdb, select=Sel())


def prepare_protein(folder, pdb_url):
    path = wget.download(pdb_url, folder)
    print()
    remove_water_and_ligands(path, path)
    return path


def write_ligand_csv(folder, protein_path, sequence, smiles_list):
    csv_path = os.path.join(folder, "protein_ligand.csv")
    with open(csv_path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["complex_name", "protein_path", "ligand_description", "protein_sequence"])
        for i, smi in enumerate(smiles_list):
            w.writerow([f"complex_{i}", protein_path, smi, sequence])
    return csv_path


def modify_yaml_config(default_path, mods, target_dir):
    with open(default_path) as f:
        cfg = deepcopy(yaml.safe_load(f))
    cfg.update(mods)
    os.makedirs(target_dir, exist_ok=True)
    out = os.path.join(target_dir, "inference_args.yaml")
    with open(out, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    return out


def run_inference(ligand_csv, out_dir, steps, samples, batch_size=10):
    cfg = modify_yaml_config(
        os.path.join(DIFFDOCK_DIR, "default_inference_args.yaml"),
        {"inference_steps": steps, "samples_per_complex": samples,
         "batch_size": batch_size, "actual_steps": steps - 1},
        out_dir,
    )
    cmd = [sys.executable, "-m", "inference", "--config", cfg,
           "--protein_ligand_csv", ligand_csv, "--out_dir", out_dir]
    proc = subprocess.Popen(cmd, cwd=DIFFDOCK_DIR, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line.rstrip())
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"DiffDock exited {proc.returncode}")


def gnina_affinity(sdf, receptor):
    cmd = [GNINA, "--local_only", "--minimize", "-r", receptor, "-l", sdf]
    if GNINA_CPU_ONLY:
        cmd.append("--cpu")
    try:
        r = subprocess.run(cmd, check=True, text=True, capture_output=True)
        m = re.search(r"Affinity:\s*([\-\.\d]+)", r.stdout + r.stderr)
        return float(m.group(1)) if m else None
    except subprocess.CalledProcessError as e:
        print(f"  gnina failed: {e.output[:200]}")
        return None


def confidence_from_filename(name):
    m = re.search(r"confidence(.+)\.sdf", name)
    try:
        return float(m.group(1)) if m else None
    except ValueError:
        return None


def collect_results(smiles_list, results_dir, receptor):
    """Returns {smiles: [{gnina_minimized_affinity, diffdock_confidence, sdf}, ...]}"""
    out = {}
    for complex_dir in sorted(glob.glob(os.path.join(results_dir, "complex_*"))):
        idx = int(os.path.basename(complex_dir).split("_")[1])
        smiles = smiles_list[idx]
        poses = []
        for sdf in glob.glob(os.path.join(complex_dir, "rank*_confidence*.sdf")):
            conf = confidence_from_filename(os.path.basename(sdf))
            if conf is None:
                continue
            aff = gnina_affinity(sdf, receptor)
            if aff is None:
                continue
            poses.append({
                "gnina_minimized_affinity": max(-1000, min(1000, round(aff, 5))),
                "diffdock_confidence":      max(-1000, min(1000, round(conf, 5))),
                "sdf":                      read_sdf_as_string(sdf),
            })
        if poses:
            out[smiles] = poses
    return out

## 8 &nbsp; Run

Checkpoints to `results/partial_*.json` after every batch and skips anything already done,
so re-running after a session dies resumes rather than restarts.

In [11]:
import json, time, os, glob, shutil
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU. DiffDock runs ~21 min/ligand on CPU and this loop would burn the\n"
        "session for nothing. Attach GPU T4 x2 and re-run the notebook from cell 1."
    )

_cap = torch.cuda.get_device_capability(0)
_sm  = f"sm_{_cap[0]}{_cap[1]}"
if _sm not in torch.cuda.get_arch_list():
    raise RuntimeError(
        f"{torch.cuda.get_device_name(0)} is {_sm}, which this torch build does not support\n"
        f"(supports: {' '.join(torch.cuda.get_arch_list())}).\n\n"
        "cuda.is_available() lies here - no kernel can launch, so DiffDock falls back to\n"
        "CPU at ~21 min/ligand or crashes outright.\n\n"
        "Fix: Session options -> Accelerator -> GPU T4 x2. Not P100."
    )

# --- load and gate ligands -------------------------------------------------
if not LIGANDS:
    print("LIGANDS is None - CONTROLS-ONLY SMOKE TEST.")
    print("Docking the two reference drugs to prove the environment works.\n")
    ligands = []
elif not os.path.exists(LIGANDS):
    raise FileNotFoundError(
        f"{LIGANDS} not found.\n"
        "Add the Dataset via '+ Add Input' in the right-hand panel, or set LIGANDS = None "
        "to run the controls-only smoke test."
    )
elif LIGANDS.endswith(".json"):
    ligands = json.load(open(LIGANDS))
    if isinstance(ligands, dict):
        ligands = list(ligands)
else:
    ligands = [l.strip().split()[0] for l in open(LIGANDS) if l.strip()]

ligands = list(dict.fromkeys(ligands))                  # dedupe, keep order
ligands = list(CONTROLS.values()) + ligands             # controls always first

kept, rejected = [], {}
for smi in ligands:
    why = gate(smi)
    (kept.append(smi) if why is None else rejected.setdefault(why, []).append(smi))

print(f"{len(ligands)} in -> {len(kept)} pass the gate, {sum(map(len, rejected.values()))} rejected")
for why, lst in sorted(rejected.items(), key=lambda kv: -len(kv[1]))[:10]:
    print(f"   {len(lst):>5}  {why}")

# --- resume ----------------------------------------------------------------
results = {}
for p in sorted(glob.glob(f"{OUT}/partial_*.json")):
    results.update(json.load(open(p)))
todo = [s for s in kept if s not in results]
print(f"\n{len(results)} already docked, {len(todo)} to go")

# --- protein prep (once) ---------------------------------------------------
work = "/kaggle/working/run"
os.makedirs(work, exist_ok=True)
receptor = glob.glob(f"{work}/*.pdb")
receptor = receptor[0] if receptor else prepare_protein(work, TARGETS[TARGET]["pdb_url"])
print("receptor:", receptor)

# --- batch loop ------------------------------------------------------------
sequence = TARGETS[TARGET]["sequence"]
t_start = time.time()

for b in range(0, len(todo), BATCH_SIZE):
    batch = todo[b:b + BATCH_SIZE]
    n = b // BATCH_SIZE
    print(f"\n=== batch {n}  ({b}-{b+len(batch)} of {len(todo)}) ===")

    bdir = f"{work}/batch_{n}"
    shutil.rmtree(bdir, ignore_errors=True)
    os.makedirs(bdir, exist_ok=True)

    try:
        ligand_csv = write_ligand_csv(bdir, receptor, sequence, batch)
        run_inference(ligand_csv, bdir, INFERENCE_STEPS, SAMPLES_PER_COMPLEX)
        got = collect_results(batch, bdir, receptor)
    except Exception as e:
        print(f"batch {n} failed: {e}")
        continue

    results.update(got)
    with open(f"{OUT}/partial_{n:04d}.json", "w") as f:
        json.dump(got, f)

    shutil.rmtree(bdir, ignore_errors=True)       # SDF text is already in the json
    done = len(results)
    rate = (time.time() - t_start) / max(done, 1)
    print(f"  {len(got)}/{len(batch)} docked | total {done} | {rate:.0f}s per ligand "
          f"| est remaining {rate*(len(todo)-b-len(batch))/3600:.1f} h")

print(f"\ndone: {len(results)} ligands")

LIGANDS is None - CONTROLS-ONLY SMOKE TEST.
Docking the two reference drugs to prove the environment works.

3 in -> 3 pass the gate, 0 rejected

0 already docked, 3 to go

receptor: /kaggle/working/run/8AFB.pdb

=== batch 0  (0-3 of 3) ===
/kaggle/working/DiffDock/utils/so3.py:60: RuntimeWarning: invalid value encountered in sqrt
  _exp_score_norms = np.sqrt(np.sum(_score_norms**2 * _pdf_vals, axis=1) / np.sum(_pdf_vals, axis=1) / np.pi)

100%|██████████| 201/201 [01:01<00:00,  3.26it/s]

100%|██████████| 201/201 [01:23<00:00,  2.41it/s]
/kaggle/working/DiffDock/utils/torus.py:38: RuntimeWarning: invalid value encountered in divide
  score_ = grad(x, sigma[:, None], N=100) / p_
/usr/lib/python3.12/ast.py:407: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/usr/lib/python3.12/ast.py:4

## 9 &nbsp; Score, sanity-check, write

Scoring is your `score_docking_results()` verbatim. The control check is the part that was
missing last time: if an approved drug does not land near the top against its own target,
the ranking is not evidence and the run should be discarded rather than reported.

In [12]:
import json
import numpy as np

MIN_AFFINITY, MAX_AFFINITY = -8, 3
MIN_CONFIDENCE, MAX_CONFIDENCE = -5, 5

def normalize(v, lo, hi):
    return (v - lo) / (hi - lo) if hi != lo else 0

def score_docking_results(poses):
    best_score, best, good = float("-inf"), {}, False
    for r in poses:
        s = (0.5 * normalize(-r["gnina_minimized_affinity"], MIN_AFFINITY, MAX_AFFINITY)
             + 0.5 * normalize(r["diffdock_confidence"], MIN_CONFIDENCE, MAX_CONFIDENCE))
        q = r["diffdock_confidence"] >= -1.6 and r["gnina_minimized_affinity"] <= -4.8
        if s > best_score:
            best_score, best, good = s, r, q
    return dict(best), best_score, good

scored = {}
for smi, poses in results.items():
    best, score, good = score_docking_results(poses)
    scored[smi] = {
        "DiffDock Confidence": best["diffdock_confidence"],
        "GNINA Minimized Affinity (Binding Energy)": best["gnina_minimized_affinity"],
        "Ligand SDF": best["sdf"],
        "Adjusted Dock Score": score,
        "Good Docking Quality": good,
    }

# --- control check ---------------------------------------------------------
ranked = sorted(scored, key=lambda s: scored[s]["GNINA Minimized Affinity (Binding Energy)"])
pdb_id = TARGETS[TARGET]["pdb_url"].rsplit("/", 1)[-1].replace(".pdb", "")
cognate_name = COGNATE.get(pdb_id, (None, None))[0]
print(f"CONTROL CHECK - {TARGET} (receptor {pdb_id}, cognate ligand {cognate_name})\n")
ok = True
for name, smi in CONTROLS.items():
    if smi not in scored:
        print(f"  {name:<11} NOT DOCKED"); ok = False; continue
    pos = ranked.index(smi) + 1
    pct = 100 * pos / len(ranked)
    aff = scored[smi]["GNINA Minimized Affinity (Binding Energy)"]
    if len(ranked) < 10:
        # smoke test: too few molecules to rank against. Just check the number is sane.
        verdict = "plausible" if aff < -5 else "SUSPECT - weak affinity for a known drug"
        print(f"  {name:<11} affinity {aff:>8.2f}  ({len(ranked)} molecules, too few to rank)  {verdict}")
        ok &= aff < -5
        continue
    is_cognate = (name == cognate_name)
    tag = " [COGNATE]" if is_cognate else " [non-cognate]"
    verdict = "ok" if pct <= 10 else ("FAILS - method broken here" if is_cognate
                                      else "weak, expected for a non-cognate reference")
    print(f"  {name:<11} affinity {aff:>8.2f}  rank {pos}/{len(ranked)} (top {pct:.0f}%){tag}  {verdict}")
    # only the cognate control gates the run
    if is_cognate:
        ok &= pct <= 10

print()
if len(ranked) < 10:
    print("Smoke test only. Environment works if both affinities came back negative.")
elif ok:
    print("Controls rank in the top decile. Proceed.")
else:
    print("A known drug did not rank. Do not report these hits - fix the scoring first.")
    print("Likely cause on G12C: DiffDock and GNINA do not model the covalent bond to Cys12.")
    print("Use a non-covalent reference binder as the control for this target.")

# --- write -----------------------------------------------------------------
with open(f"{OUT}/{TARGET}_raw_poses.json", "w") as f:
    json.dump(results, f)
with open(f"{OUT}/{TARGET}_scored.json", "w") as f:
    json.dump(scored, f, indent=2)

print(f"\nwrote {OUT}/{TARGET}_scored.json  ({len(scored)} ligands)")
print("Download it, then run adapt_results.py locally to merge into your results.json format.")

CONTROL CHECK - KRAS_G12C (receptor 8AFB, cognate ligand BI-0474)

  BI-0474     affinity    -8.07  (3 molecules, too few to rank)  plausible
  Sotorasib   affinity    -7.78  (3 molecules, too few to rank)  plausible
  Adagrasib   affinity    -8.16  (3 molecules, too few to rank)  plausible

Smoke test only. Environment works if both affinities came back negative.

wrote /kaggle/working/results/KRAS_G12C_scored.json  (3 ligands)
Download it, then run adapt_results.py locally to merge into your results.json format.


## 9b &nbsp; Diagnostic (only if nothing docked)

In [13]:
# Diagnostic. Run this when the batch loop produces nothing.
# Docks ONE ligand with cleanup disabled so the failure is visible.
import os, glob, shutil, subprocess, re, sys

print(f"results dict: {len(results)} entries\n")

dbg = "/kaggle/working/debug"
shutil.rmtree(dbg, ignore_errors=True)
os.makedirs(dbg, exist_ok=True)

one = list(CONTROLS.values())[:1]
seq = TARGETS[TARGET]["sequence"]

print("1. receptor:", receptor, os.path.exists(receptor))
print("   size:", os.path.getsize(receptor) if os.path.exists(receptor) else "MISSING")

print("\n2. DiffDock on 1 ligand (full output):")
csv_path = write_ligand_csv(dbg, receptor, seq, one)
try:
    run_inference(csv_path, dbg, INFERENCE_STEPS, SAMPLES_PER_COMPLEX)
except Exception as e:
    print("   DiffDock raised:", e)

print("\n3. output tree:")
for root, dirs, files in os.walk(dbg):
    lvl = root.replace(dbg, "").count(os.sep)
    print("   " + "  " * lvl + os.path.basename(root) + "/")
    for f in files[:8]:
        print("   " + "  " * (lvl + 1) + f)

sdfs = glob.glob(os.path.join(dbg, "complex_*", "rank*_confidence*.sdf"))
print(f"\n   pose SDFs found: {len(sdfs)}")
if not sdfs:
    print("   -> DiffDock produced no poses. The failure is in its output above.")

if sdfs:
    print("\n4. GNINA on one pose:")
    cmd = [GNINA, "--local_only", "--minimize", "-r", receptor, "-l", sdfs[0]]
    if GNINA_CPU_ONLY:
        cmd.append("--cpu")
    r = subprocess.run(cmd, capture_output=True, text=True)
    print("   exit code:", r.returncode)
    out = (r.stdout + r.stderr).strip()
    for line in out.splitlines()[-25:]:
        print("     ", line[:120])
    m = re.search(r"Affinity:\s*([\-\.\d]+)", out)
    print("\n   parsed affinity:", m.group(1) if m else "NOT FOUND  <-- regex matched nothing")

results dict: 3 entries

1. receptor: /kaggle/working/run/8AFB.pdb True
   size: 106928

2. DiffDock on 1 ligand (full output):
/usr/lib/python3.12/ast.py:407: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/usr/lib/python3.12/ast.py:407: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/usr/lib/python3.12/ast.py:407: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/usr/lib/python3.12/

## 10 &nbsp; Top hits

In [14]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

rows = []
inv = {v: k for k, v in CONTROLS.items()}
for smi, r in scored.items():
    m = Chem.MolFromSmiles(smi)
    rows.append({
        "control": inv.get(smi, ""),
        "affinity": r["GNINA Minimized Affinity (Binding Energy)"],
        "confidence": r["DiffDock Confidence"],
        "score": round(r["Adjusted Dock Score"], 3),
        "good": r["Good Docking Quality"],
        "MW": round(Descriptors.MolWt(m), 1),
        "arom": rdMolDescriptors.CalcNumAromaticRings(m),
        "SMILES": smi[:60],
    })

if not scored:
    print("Nothing docked. Run the diagnostic cell above.")
else:
    df = pd.DataFrame(rows).sort_values("affinity")
    print(f"{df['good'].sum()} of {len(df)} pass the docking-quality filter\n")
    display(df.head(30))

1 of 3 pass the docking-quality filter



,control,affinity,confidence,score,good,MW,arom,SMILES
2,Adagrasib,-8.16353,-1.33,0.918,True,604.1,3,[H][C@@]1(COC2=NC3=C(CCN(C3)C3=CC=CC4=C3C(Cl)=...
0,BI-0474,-8.06555,-3.52,0.804,False,587.8,3,C[C@H]1CN(CCN1c2cc(cc(n2)c3nc(on3)[C@]4(CCCc5c...
1,Sotorasib,-7.77665,-4.72,0.731,False,560.6,4,CC(C)C1=NC=CC(C)=C1N1C(=O)N=C(N2CCN(C[C@@H]2C)...
